# Question 7 - ViT Patching (Fashion-MNIST, P = 7, D = 32)
1. **N = (28/7) x (28/7) = 4 x 4 = 16 patches**
2. Each patch: 7x7x1 = 49 values, projected to D = 32, giving (16, 32). Prepending 1 [class] token gives **(17, 32)**, or **(B, 17, 32)** with a batch.

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [ ]:
(x_train, _), _ = keras.datasets.fashion_mnist.load_data()
images = (x_train[:8].astype("float32") / 255.0)[..., None]     # (8, 28, 28, 1)
P, D = 7, 32

In [ ]:
class PatchEmbedding(layers.Layer):
    def __init__(self, patch_size, dim):
        super().__init__()
        self.p = patch_size
        self.proj = layers.Dense(dim)
        self.cls_token = self.add_weight(shape=(1, 1, dim), initializer="zeros", name="cls")

    def call(self, x):
        b = tf.shape(x)[0]
        patches = tf.image.extract_patches(x, sizes=[1, self.p, self.p, 1], strides=[1, self.p, self.p, 1],
                                           rates=[1, 1, 1, 1], padding="VALID")
        patches = tf.reshape(patches, (b, -1, self.p * self.p * x.shape[-1]))
        print("flattened patches:", patches.shape)
        tokens = self.proj(patches)
        print("after projection :", tokens.shape)
        cls = tf.tile(self.cls_token, [b, 1, 1])
        return tf.concat([cls, tokens], axis=1)

seq = PatchEmbedding(P, D)(images)
print("with [class] token:", seq.shape)
print("N =", (28 // P) ** 2)